In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use('Agg')  # non-interactive backend, safe for batch processing
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from scipy.signal import butter, filtfilt
from scipy.ndimage import gaussian_filter1d
import h5py

from config import dir_config, ephys_config

In [ ]:
compiled_dir = Path(dir_config.data.compiled)
sorted_dir = Path(dir_config.data.sorting)
processed_dir = Path(dir_config.data.processed)

In [ ]:
session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))

### Helper Function

#### Loading data

In [ ]:
def load_phy_data(session_id):
    session_row = session_metadata[session_metadata['session_id'] == session_id]
    if session_row.empty or pd.isna(session_row['sort_folder'].values[0]):
        print(f"  No sort folder for session {session_id}, skipping.")
        return None
    
    sort_folder = session_row['sort_folder'].values[0]
    kl_dir = Path(sorted_dir, session_id, sort_folder)
    if not kl_dir.exists():
        print(f"  Sort folder not found: {kl_dir}, skipping.")
        return None

    sorting_data = {
        "spike_times":    np.load(kl_dir / "spike_times.npy").flatten(),
        "spike_clusters": np.load(kl_dir / "spike_clusters.npy").flatten(),
        "amplitudes":     np.load(kl_dir / "amplitudes.npy").flatten(),
        "chan_pos":        np.load(kl_dir / "channel_positions.npy"),  # (n_channels, 2), chanMap order
    }

    # channel_map.npy: chanMap index -> hardware channel number (= column in raw binary)
    chan_map_path = kl_dir / "channel_map.npy"
    chan_map = np.load(chan_map_path).flatten() if chan_map_path.exists() else None
    sorting_data['chan_map'] = chan_map
    # Inverse: hardware channel number -> chanMap index (row in chan_pos)
    sorting_data['hw_to_chanmap_idx'] = (
        {int(hw): idx for idx, hw in enumerate(chan_map)} if chan_map is not None else None
    )

    # Peak channel per cluster from cluster_info.tsv (ch = hardware channel number)
    info_path = kl_dir / 'cluster_info.tsv'
    if info_path.exists():
        info = pd.read_csv(info_path, sep='\t')
        sorting_data['cluster_ch'] = dict(zip(info['cluster_id'], info['ch'].astype(int)))
    else:
        sorting_data['cluster_ch'] = {}

    # Sample rate and recording parameters from params.py
    params_path = kl_dir / 'params.py'
    params = {}
    if params_path.exists():
        with open(params_path) as f:
            exec(f.read(), params)
    sorting_data['sample_rate'] = int(params.get('sample_rate', 30000))
    sorting_data['n_channels']  = int(params.get('n_channels_dat', 0))
    sorting_data['dtype']       = params.get('dtype', 'int16')

    return sorting_data

def load_plexon_data(session_id):
    """
    Load sorting data for sessions sorted by Plexon Offline Sorter (sort_folder is NaN).

    Reads spike timestamps, unit assignments, and pre-stored waveform snippets
    from the NEV file (_sort.mat). Amplitude is computed as peak-to-trough per waveform.

    Returns a dict compatible with run_qc, with an extra key 'wf_by_cluster':
        {cluster_id: np.ndarray of shape (n_spikes, 48)} — pre-extracted waveforms.
    Channel info (Blackrock_channel) and cluster IDs come from neuron_metadata.
    """

    mat_path = Path(sorted_dir, session_id, f"{session_id}_sort.mat")
    if not mat_path.exists():
        print(f"  NEV file not found: {mat_path}, skipping.")
        return None

    with h5py.File(str(mat_path), 'r') as f:
        sample_rate = int(f['NEV/MetaTags/SampleRes'][()].flat[0])
        spike_times    = f['NEV/Data/Spikes/TimeStamp'][()].flatten().astype(np.uint32)
        spike_channels = f['NEV/Data/Spikes/Electrode'][()].flatten().astype(np.uint16)
        spike_units    = f['NEV/Data/Spikes/Unit'][()].flatten().astype(np.uint8)
        waveforms      = f['NEV/Data/Spikes/Waveform'][()]   # (n_spikes, 48) int16

    # Amplitude = peak-to-trough per spike
    amplitudes = waveforms.max(axis=1).astype(float) - waveforms.min(axis=1).astype(float)

    # Collect pre-extracted waveforms per cluster (for the relevant channel only)
    session_neurons = neuron_metadata[neuron_metadata['session_id'] == session_id]
    hw_channel = int(session_neurons['Blackrock_channel'].iloc[0])
    cluster_ids = session_neurons['cluster'].values.astype(int)

    ch_mask = spike_channels == hw_channel
    wf_by_cluster = {}
    for cid in cluster_ids:
        mask = ch_mask & (spike_units == cid)
        wf_by_cluster[cid] = waveforms[mask]   # (n_spikes_for_unit, 48)

    # cluster_ch: cluster_id -> hardware channel (same for all units here)
    cluster_ch = {int(cid): hw_channel for cid in cluster_ids}

    # Minimal chan_pos / chan_map for single-electrode compatibility
    chan_pos = np.array([[0.0, 0.0]])   # single channel, arbitrary position
    chan_map = np.array([hw_channel])   # chanMap index 0 -> hw channel
    hw_to_chanmap_idx = {hw_channel: 0}

    return {
        "spike_times":        spike_times[ch_mask],
        "spike_clusters":     spike_units[ch_mask],
        "amplitudes":         amplitudes[ch_mask],
        "wf_by_cluster":      wf_by_cluster,
        "cluster_ch":         cluster_ch,
        "chan_pos":            chan_pos,
        "chan_map":            chan_map,
        "hw_to_chanmap_idx":   hw_to_chanmap_idx,
        "sample_rate":        sample_rate,
        "n_channels":         1,
        "dtype":              "int16",
    }

def load_trial_events(session_id, sample_rate=30000, valid_only=True, GP_only=True):
    filename = f"{session_id}_timestamps_cleaned.csv"
    timestamps = pd.read_csv(Path(compiled_dir, session_id, filename), index_col=None)
    if valid_only:
        timestamps = timestamps[~np.isnan(timestamps['response_onset'])]
    if GP_only:
        timestamps = timestamps[~np.isnan(timestamps['stimulus_onset'])]

    event_cols = [
        'fixation_onset', 'target_onset', 'stimulus_onset',
        'go_onset', 'response_onset', 'trial_offset',
    ]
    events = {}
    for col in event_cols:
        if col in timestamps.columns:
            events[col] = pd.to_numeric(timestamps[col]) / sample_rate  # samples -> seconds
        else:
            print(f"  Warning: column '{col}' not found in trial file, skipping.")
    return events


#### Quality Metrics

In [ ]:
def compute_fr_similarity(spike_times_sec, trial_times_sec, before=2.0):
    """
    Firing rate stability score anchored to fixation_onset.
    Splits trials in half, computes mean FR in the pre-fixation window for each half.

    Score = 1 - 2 * |FR1 - FR2| / (FR1 + FR2)
      1   = perfectly stable across the session
      0   = FR in one half is double the other
     <0   = severe drift (one half near-silent)
     NaN  = unit silent in at least one half
    """
    valid = trial_times_sec[~np.isnan(trial_times_sec)]
    if len(valid) < 2:
        return np.nan

    st   = np.sort(spike_times_sec)
    half = len(valid) // 2

    def mean_fr(trial_subset):
        hi = np.searchsorted(st, trial_subset,          side='right')
        lo = np.searchsorted(st, trial_subset - before, side='left')
        return np.mean(hi - lo)

    fr1   = mean_fr(valid[:half])
    fr2   = mean_fr(valid[half:])
    denom = fr1 + fr2
    if denom == 0:
        return np.nan
    return 1 - 2 * abs(fr1 - fr2) / denom


def compute_isi_violations(spike_times_sec, refractory_period_ms=1.5):
    """
    Refractory period violation ratio.
    Returns fraction of ISIs below the refractory period threshold.
    Pass threshold: < 2%.
    """
    if len(spike_times_sec) < 2:
        return np.nan
    isis = np.diff(np.sort(spike_times_sec))
    rp_s = refractory_period_ms / 1000.0
    return np.sum(isis < rp_s) / len(isis)


def compute_noise_cutoff(amplitudes, n_bins=100, percent_threshold=10.0):
    """
    Noise cutoff metric (IBL pipeline).
    Checks if the amplitude histogram is truncated at the low end.
    metric_value = lowest bin count as % of peak bin count.
    Pass if metric_value < percent_threshold (default 10%).
    """
    if len(amplitudes) < 10:
        return False, np.nan
    counts, _ = np.histogram(amplitudes, bins=n_bins)
    if counts.max() == 0:
        return False, np.nan
    low_bin_pct = counts[0] / counts.max() * 100
    return low_bin_pct < percent_threshold, low_bin_pct


def compute_acg(spike_times_sec, bin_ms=0.5, max_lag_ms=50):
    """
    Autocorrelogram via FFT autocorrelation on a binned spike train.
    O(N log N) in recording length — orders of magnitude faster than the
    O(n_spikes²) loop for large spike counts.
    Returns (bin_centers_ms, counts).
    """
    if len(spike_times_sec) < 2:
        return np.array([]), np.array([])

    bin_s      = bin_ms / 1000.0
    n_lag_bins = int(round(max_lag_ms / bin_ms))

    # Build binned spike train over the full recording duration
    t_min     = spike_times_sec.min()
    t_max     = spike_times_sec.max()
    n_total   = int(np.ceil((t_max - t_min) / bin_s)) + 1
    train     = np.zeros(n_total, dtype=np.float32)
    spike_idx = np.round((spike_times_sec - t_min) / bin_s).astype(int)
    np.add.at(train, spike_idx, 1)

    # FFT-based autocorrelation
    from scipy.signal import fftconvolve
    acg_full = fftconvolve(train, train[::-1], mode='full')
    center   = len(train) - 1
    acg      = acg_full[center - n_lag_bins : center + n_lag_bins + 1].copy()
    acg[n_lag_bins] = 0  # remove zero-lag (spike with itself)

    bin_centers = np.arange(-n_lag_bins, n_lag_bins + 1) * bin_ms
    return bin_centers, acg


def sliding_rp_confidence(spike_times_sec, sample_rate, rp_range_ms=None, cont_range=None):
    """
    Sliding refractory period confidence matrix (simplified IBL/Llobet approach).
    Returns (conf_matrix, rp_range_ms, cont_range).
    conf_matrix shape: (n_cont, n_rp), values 0-100.
    """
    from scipy.stats import poisson

    if rp_range_ms is None:
        rp_range_ms = np.arange(0.5, 5.1, 0.5)
    if cont_range is None:
        cont_range = np.arange(0, 31, 1)

    n_spikes = len(spike_times_sec)
    if n_spikes < 2:
        return np.zeros((len(cont_range), len(rp_range_ms))), rp_range_ms, cont_range

    duration    = spike_times_sec.max() - spike_times_sec.min()
    firing_rate = n_spikes / duration if duration > 0 else 0
    isis        = np.diff(np.sort(spike_times_sec))

    conf_matrix = np.zeros((len(cont_range), len(rp_range_ms)))
    for i, rp_ms in enumerate(rp_range_ms):
        rp_s     = rp_ms / 1000.0
        n_viol   = np.sum(isis < rp_s)
        # Vectorise over the contamination axis — one poisson.cdf call per rp value
        expected = (cont_range / 100.0) * firing_rate * rp_s * n_spikes * 2
        with np.errstate(invalid='ignore'):
            conf_col = poisson.cdf(n_viol, expected) * 100
        conf_col[expected == 0] = 100.0 if n_viol == 0 else 0.0
        conf_matrix[:, i] = conf_col

    return conf_matrix, rp_range_ms, cont_range


#### Waveform Extraction from raw binary data

In [ ]:
WF_PRE_SAMPLES  = 8
WF_POST_SAMPLES = 40
WF_N_SAMPLES    = WF_PRE_SAMPLES + WF_POST_SAMPLES  # 48  (used for Plexon NEV waveforms)


def get_neighboring_channels(peak_ch, chan_pos, n=4):
    """
    Return chanMap indices of peak channel + (n-1) nearest neighbors by probe distance.
    peak_ch and return values are chanMap indices; chan_pos is in chanMap order.
    """
    d = np.sqrt(((chan_pos - chan_pos[peak_ch]) ** 2).sum(axis=1))
    d[peak_ch] = np.inf
    nearest = np.argsort(d)[:n - 1]
    return np.concatenate([[peak_ch], nearest])


def extract_waveforms(session_id, spike_times_samples, n_channels, sample_rate,
                      dtype='int16', n_wf=200, hp_cutoff=300, channels=None, chan_map=None):
    """
    Extract and high-pass filter waveforms from raw binary file.
    Window: ±1 ms centred on the spike timestamp (30 pre + 30 post = 60 samples at 30 kHz).

    channels: chanMap indices of channels to extract (from get_neighboring_channels).
    chan_map: array mapping chanMap index -> hardware channel number (raw binary column).
             If None, channels are used directly as column indices (assumes identity mapping).
    Returns array of shape (n_wf, n_samples, len(channels)), or None if unavailable.
    """
    raw_path = Path(sorted_dir, session_id, f"{session_id}.bin")

    if not raw_path.exists():
        print(f"  Warning: raw data file not found: {raw_path}")
        return None

    n_samples = int(round(sample_rate * 2 / 1000))  # 2 ms total window
    half      = n_samples // 2

    if channels is not None and chan_map is not None:
        hw_channels = chan_map[channels]
    else:
        hw_channels = channels

    dt = np.dtype(dtype)

    n_spikes = len(spike_times_samples)
    idx      = np.random.choice(n_spikes, min(n_wf, n_spikes), replace=False)
    selected = np.sort(spike_times_samples[idx])

    waveforms = []
    with open(raw_path, 'rb') as f:
        for t in selected:
            start = int(t) - half
            if start < 0:
                continue
            f.seek(start * n_channels * dt.itemsize)
            chunk = np.frombuffer(
                f.read(n_samples * n_channels * dt.itemsize), dtype=dt
            )
            if len(chunk) != n_samples * n_channels:
                continue
            wf = chunk.reshape(n_samples, n_channels)
            waveforms.append(wf[:, hw_channels] if hw_channels is not None else wf)

    if not waveforms:
        return None

    wf_array = np.array(waveforms, dtype=float)  # (n_wf, n_samples, n_ch_out)

    b, a = butter(2, hp_cutoff / (sample_rate / 2), btype='high', output='ba')
    for w in range(wf_array.shape[0]):
        for ch in range(wf_array.shape[2]):
            wf_array[w, :, ch] = filtfilt(b, a, wf_array[w, :, ch])

    return wf_array


#### Plotting

In [ ]:
PASS_COLOR = np.array([34, 177, 76]) / 255
FAIL_COLOR = np.array([220, 50, 50]) / 255
NAN_COLOR  = np.array([200, 180, 0]) / 255

# (event_key, label, before_s, after_s)
RASTER_EVENTS = [
    ('target_onset',   'Target onset',   0.2,  0.3),
    ('stimulus_onset', 'GP Onset',       0.2,  0.5),
    ('response_onset', 'Response onset', 0.4,  0.1),
]


def fmt_pass(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return '~'
    return 'PASS' if val else 'FAIL'


def unit_color(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return NAN_COLOR
    return PASS_COLOR if val else FAIL_COLOR


def plot_raster_panel(ax, spike_times_sec, event_times_sec, before, after, event_label):
    spike_times_sec  = np.asarray(spike_times_sec,  dtype=float)
    event_times_sec  = np.asarray(event_times_sec,  dtype=float)

    valid_mask  = ~np.isnan(event_times_sec)
    valid_times = event_times_sec[valid_mask]
    n_trials    = len(valid_times)

    if n_trials == 0:
        ax.text(0.5, 0.5, f'No valid trials\nfor {event_label}',
                ha='center', va='center', transform=ax.transAxes, color='gray')
        ax.set_title(event_label, fontsize=9)
        return

    bin_s     = 0.001  # 1 ms bins
    bins      = np.arange(-before, after + bin_s, bin_s)
    n_bins    = len(bins) - 1
    psth      = np.zeros((n_trials, n_bins))
    before_ms = before * 1000
    after_ms  = after  * 1000

    for i, t0 in enumerate(valid_times):
        rel    = spike_times_sec - t0
        spk    = rel[(rel >= -before) & (rel < after)]
        if len(spk):
            ax.vlines(spk * 1000, i + 0.5, i + 1.5,
                      color='black', linewidth=0.4, alpha=0.7)
        psth[i], _ = np.histogram(rel, bins=bins)
        psth[i] = psth[i].astype(float) / bin_s  # Hz

    ax.axvline(0, color='cyan', linewidth=1.2, linestyle='--')
    ax.set_xlim(-before_ms, after_ms)
    ax.set_ylim(0, n_trials)
    ax.set_xlabel('Time (ms)', fontsize=8)
    ax.set_ylabel('Trial #', fontsize=8)
    ax.tick_params(labelsize=7)

    mean_fr = gaussian_filter1d(psth.mean(axis=0), sigma=3)
    t_axis  = (bins[:-1] + bins[1:]) / 2 * 1000  # seconds -> ms
    ax2 = ax.twinx()
    ax2.plot(t_axis, mean_fr, color='royalblue', linewidth=1.2, alpha=0.6)
    ax2.set_ylabel('Mean FR (Hz)', color='royalblue', fontsize=7)
    ax2.tick_params(axis='y', colors='royalblue', labelsize=7)

    ax.set_title(f'{event_label}  ({n_trials} trials)', fontsize=9)


def plot_unit(session_id, unit_idx, neuron_id, cluster_id, spike_times_sec, amplitudes,
              trial_events, fr_similarity, isi_ratio, nc_pass, nc_value,
              acg_bins, acg_counts, conf_matrix, rp_range_ms, cont_range,
              wf_array, channels, chan_pos, chan_map=None, wf_time_ms=None,
              before=2.0, after=2.0, n_units_total=1, output_dir=None):
    """
    Layout (3 rows × 5 cols):
      Row 0: raster×3  |  FR across trials (cols 3-4, 2-wide)
      Row 1: amp_scatter  amp_hist  wf1  wf2  summary
      Row 2: acg  sliding_rp  wf3  wf4  (empty)

    wf_time_ms: 1-D array of waveform time values in ms, with t=0 at the reference point
                (threshold crossing for Plexon; spike peak for Phy ±1 ms snippets).
                If None, falls back to linspace(-1, 1) over the snippet length.
    """
    spike_times_sec = np.asarray(spike_times_sec, dtype=float)
    amplitudes      = np.asarray(amplitudes,      dtype=float)

    stable   = (not np.isnan(fr_similarity)) and fr_similarity > 0
    isi_pass = (not np.isnan(isi_ratio)) and isi_ratio < 0.02
    overall  = stable and isi_pass and nc_pass

    fig = plt.figure(figsize=(24, 14), facecolor='white')
    gs  = gridspec.GridSpec(3, 5, figure=fig, hspace=0.55, wspace=0.42)

    stab_str = ('Stable' if stable
                else ('NaN FR' if np.isnan(fr_similarity) else 'Unstable'))
    line1 = (f"Session {session_id}  |  Neuron {neuron_id}  |  Cluster {cluster_id}  |  "
             f"Overall: {fmt_pass(overall)}")
    line2 = (f"Stability: {fmt_pass(stable)} ({stab_str})  |  "
             f"ISI violations: {fmt_pass(isi_pass)} ({isi_ratio * 100:.2f}%)  |  "
             f"Noise cutoff: {fmt_pass(nc_pass)} ({nc_value:.1f}%)")
    gt = fig.suptitle(f"{line1}\n{line2}", fontsize=11, fontweight='bold', y=0.99)
    gt.set_color(unit_color(overall))

    # --- Row 0: rasters (cols 0-2) + FR across trials (cols 3-4, 2-wide) ---
    for col_i, (event_key, event_label, ev_before, ev_after) in enumerate(RASTER_EVENTS):
        ax = fig.add_subplot(gs[0, col_i])
        if event_key in trial_events:
            plot_raster_panel(ax, spike_times_sec, trial_events[event_key],
                              ev_before, ev_after, event_label)
        else:
            ax.text(0.5, 0.5, f'{event_label}\nnot in trial file',
                    ha='center', va='center', transform=ax.transAxes, color='gray')
            ax.set_title(event_label, fontsize=9)

    ax_frt    = fig.add_subplot(gs[0, 3:5])   # spans cols 3-4
    fix_times = trial_events.get('fixation_onset', np.array([]))
    valid_fix = fix_times[~np.isnan(fix_times)] if len(fix_times) else np.array([])

    if len(valid_fix) > 0:
        st_sorted = np.sort(spike_times_sec)
        hi       = np.searchsorted(st_sorted, valid_fix,          side='right')
        lo       = np.searchsorted(st_sorted, valid_fix - before, side='left')
        trial_fr = (hi - lo) / before

        ax_frt.plot(trial_fr, 'k.', markersize=3, alpha=0.5)
        win     = min(20, len(trial_fr))
        mov_avg = np.convolve(trial_fr, np.ones(win) / win, mode='valid')
        ax_frt.plot(np.arange(win // 2, win // 2 + len(mov_avg)), mov_avg,
                    color='royalblue', linewidth=1.8)
        ax_frt.axhline(trial_fr.mean(), color='royalblue', linestyle='--', linewidth=1)
        ax_frt.axvline(len(trial_fr) // 2, color='gray', linestyle=':', linewidth=1)

    stab_val_str = f'{fr_similarity:.3f}' if not np.isnan(fr_similarity) else 'NaN'
    ax_frt.set_xlabel('Trial number', fontsize=8)
    ax_frt.set_ylabel('Mean FR (Hz)', fontsize=8)
    ax_frt.set_title(
        f'FR across trials  |  stability = {stab_val_str}\n({before}s pre-fixation window)',
        fontsize=9,
    )
    ax_frt.tick_params(labelsize=7)

    # --- Row 1: amp scatter, amp hist, wf1, wf2, summary ---
    ax_amp = fig.add_subplot(gs[1, 0])
    ax_amp.scatter(spike_times_sec / 60, amplitudes, s=1, c='black', alpha=0.25)
    ax_amp.set_xlabel('Time in session (min)', fontsize=8)
    ax_amp.set_ylabel('Template amplitude (a.u.)', fontsize=8)
    ax_amp.set_title('Spike amplitudes over time', fontsize=9)
    ax_amp.tick_params(labelsize=7)

    ax_hist = fig.add_subplot(gs[1, 1])
    ax_hist.hist(amplitudes, bins=100, orientation='horizontal',
                 color='steelblue', edgecolor='none', alpha=0.85)
    ax_hist.set_xlabel('Count', fontsize=8)
    ax_hist.set_ylabel('Amplitude (a.u.)', fontsize=8)
    nc_col = PASS_COLOR if nc_pass else FAIL_COLOR
    ax_hist.set_title(f'Amplitude histogram\nLow bin: {nc_value:.1f}% of peak',
                      color=nc_col, fontsize=9)
    ax_hist.tick_params(labelsize=7)

    ax_sum = fig.add_subplot(gs[1, 4])   # row 1, below the FR panel
    ax_sum.axis('off')
    summary_text = (
        f"Cluster {cluster_id}\n"
        f"N spikes: {len(spike_times_sec):,}\n"
        f"Mean FR:  {len(spike_times_sec) / (spike_times_sec.max() - spike_times_sec.min()):.2f} Hz\n\n"
        f"FR similarity:  {fr_similarity:.3f}\n"
        f"ISI viol:       {isi_ratio * 100:.2f}%\n"
        f"Noise cutoff:   {nc_value:.1f}%\n\n"
        f"Stability:  {fmt_pass(stable)}\n"
        f"ISI:        {fmt_pass(isi_pass)}\n"
        f"Noise cut:  {fmt_pass(nc_pass)}\n\n"
        f"OVERALL:    {fmt_pass(overall)}"
    )
    ax_sum.text(0.05, 0.95, summary_text, transform=ax_sum.transAxes,
                fontsize=9, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))

    # --- Row 2: acg, sliding_rp ---
    ax_acg = fig.add_subplot(gs[2, 0])
    if len(acg_bins) > 0:
        ax_acg.bar(acg_bins, acg_counts, width=(acg_bins[1] - acg_bins[0]),
                   color='steelblue', edgecolor='none')
        ax_acg.axvspan(-1.5, 1.5, color='red', alpha=0.15)
    ax_acg.set_xlabel('Lag (ms)', fontsize=8)
    ax_acg.set_ylabel('Count', fontsize=8)
    ax_acg.set_title(f'Autocorrelogram\nISI violations: {isi_ratio * 100:.2f}%',
                     color=unit_color(isi_pass), fontsize=9)
    ax_acg.tick_params(labelsize=7)

    ax_rp = fig.add_subplot(gs[2, 1])
    cmap  = LinearSegmentedColormap.from_list(
        'conf', ['#1a1a2e', '#16213e', '#0f3460', '#53d8fb']
    )
    im = ax_rp.imshow(conf_matrix, aspect='auto', origin='upper', cmap=cmap,
                      vmin=0, vmax=100,
                      extent=[rp_range_ms[0], rp_range_ms[-1],
                               cont_range[-1], cont_range[0]])
    ax_rp.axhline(10, color='red', linewidth=1)
    plt.colorbar(im, ax=ax_rp, label='Confidence (%)')
    ax_rp.set_xlabel('Refractory period (ms)', fontsize=8)
    ax_rp.set_ylabel('Contamination (%)', fontsize=8)
    ax_rp.set_title('Sliding RP confidence', fontsize=9)
    ax_rp.tick_params(labelsize=7)

    # --- Waveform panels: rows 1-2, cols 2-3 ---
    wf_positions = [(1, 2), (1, 3), (2, 2), (2, 3)]
    n_wf_panels  = len(wf_positions)

    if wf_array is not None and channels is not None:
        n_samples = wf_array.shape[1]
        # Plexon: caller passes wf_time_ms aligned to threshold crossing (t=0).
        # Phy: None -> symmetric ±1 ms window centred on spike peak.
        t_wf = wf_time_ms if wf_time_ms is not None else np.linspace(-1, 1, n_samples)

        for plot_i in range(n_wf_panels):
            r, c  = wf_positions[plot_i]
            ax_wf = fig.add_subplot(gs[r, c])

            if plot_i < len(channels):
                ch    = channels[plot_i]
                wf_ch = wf_array[:, :, plot_i]

                ax_wf.plot(t_wf, wf_ch.T, color='gray', linewidth=0.7, alpha=0.25)
                ax_wf.plot(t_wf, wf_ch.mean(axis=0), color='black', linewidth=1.8)\

                d_um  = np.sqrt(((chan_pos[ch] - chan_pos[channels[0]]) ** 2).sum())
                hw_ch = int(chan_map[ch]) if chan_map is not None else ch
                label = f'Ch {hw_ch}  (peak)' if plot_i == 0 else f'Ch {hw_ch}  ({d_um:.0f} um)'
                ax_wf.set_title(label, fontsize=8)
                ax_wf.set_xlabel('Time re: threshold (ms)', fontsize=7)
                ax_wf.tick_params(labelsize=7)
            else:
                ax_wf.text(0.5, 0.5, 'Single electrode',
                           ha='center', va='center', fontsize=8, color='gray',
                           transform=ax_wf.transAxes)
                ax_wf.axis('off')
    else:
        for plot_i in range(n_wf_panels):
            r, c  = wf_positions[plot_i]
            ax_wf = fig.add_subplot(gs[r, c])
            ax_wf.text(0.5, 0.5, 'Raw data unavailable',
                       ha='center', va='center', fontsize=8, color='gray',
                       transform=ax_wf.transAxes)
            ax_wf.axis('off')

    plt.show()
    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        base = os.path.join(output_dir, f'Neuron_{neuron_id:04d}_Cluster_{cluster_id}')
        fig.savefig(base + '.png', dpi=150, bbox_inches='tight')
        fig.savefig(base + '.pdf', bbox_inches='tight')

    # plt.close(fig)


## Main Code

In [ ]:
def run_qc(session_id, output_dir=None, before=2.0, after=2.0):

    session_row = session_metadata[session_metadata['session_id'] == session_id]
    is_plexon = session_row.empty or pd.isna(session_row['sort_folder'].values[0])

    if is_plexon:
        print(f"Loading Plexon data from: {session_id}")
        d = load_plexon_data(session_id)
    else:
        print(f"Loading Phy data from: {session_id}")
        d = load_phy_data(session_id)

    if d is None:
        return pd.DataFrame()
    fs = d['sample_rate']
    print(f"  Sample rate: {fs} Hz")

    print(f"Loading trial events from: {session_id}")
    trial_events = load_trial_events(session_id, fs)
    print(f"  Events loaded: {list(trial_events.keys())}")
    for k, v in trial_events.items():
        n_valid = int(np.sum(~np.isnan(v)))
        print(f"    {k}: {n_valid} valid trials out of {len(v)}")

    spike_times_sec = d['spike_times'] / fs

    session_neurons = neuron_metadata[neuron_metadata['session_id'] == session_id]
    if len(session_neurons) == 0:
        print(f"  No neurons found in neuron_metadata for session {session_id}.")
        return pd.DataFrame()
    cluster_ids = session_neurons['cluster'].values
    neuron_ids  = session_neurons['neuron_id'].values

    print(f"  Processing {len(cluster_ids)} units\n")

    summary_rows = []

    for unit_idx, (cluster_id, neuron_id) in enumerate(zip(cluster_ids, neuron_ids)):
        print(f"  Neuron {neuron_id}  Cluster {cluster_id}", end='  ')

        mask = d['spike_clusters'] == cluster_id
        st   = spike_times_sec[mask]
        amps = d['amplitudes'][mask]

        if len(st) < 10:
            print("SKIP (fewer than 10 spikes)")
            continue

        fix_times = trial_events.get('fixation_onset', np.array([np.nan]))
        fr_sim    = compute_fr_similarity(st, fix_times, before=before)
        isi_r     = compute_isi_violations(st)
        nc_pass, nc_val = compute_noise_cutoff(amps)
        acg_bins, acg_counts = compute_acg(st)
        conf_mat, rp_ax, cont_ax = sliding_rp_confidence(st, fs)

        # Waveform: use pre-extracted snippets (Plexon) or extract from raw binary (Phy)
        if 'wf_by_cluster' in d:
            wf_raw = d['wf_by_cluster'].get(int(cluster_id))
            if wf_raw is not None and len(wf_raw) > 0:
                idx = np.random.choice(len(wf_raw), min(200, len(wf_raw)), replace=False)
                wf_array = wf_raw[idx, :, np.newaxis].astype(float)  # (n_wf, 48, 1)
                b, a = butter(2, 300 / (fs / 2), btype='high', output='ba')
                for w in range(wf_array.shape[0]):
                    wf_array[w, :, 0] = filtfilt(b, a, wf_array[w, :, 0])
            else:
                wf_array = None
            channels = np.array([0])   # single chanMap index
            # Plexon NEV waveforms: 8 samples pre-threshold, remaining samples post.
            # t=0 is the threshold crossing (sample index 8).
            wf_time_ms = (np.arange(WF_N_SAMPLES) - WF_PRE_SAMPLES) / fs * 1000
        else:
            peak_hw_ch = d['cluster_ch'].get(int(cluster_id))
            if peak_hw_ch is not None and d['hw_to_chanmap_idx'] is not None:
                peak_ch = d['hw_to_chanmap_idx'].get(peak_hw_ch)
            else:
                peak_ch = peak_hw_ch
            channels = get_neighboring_channels(peak_ch, d['chan_pos'], n=4) if peak_ch is not None else None
            wf_array = extract_waveforms(session_id, d['spike_times'][mask], d['n_channels'], fs,
                                         dtype=d['dtype'], channels=channels, chan_map=d['chan_map'])
            wf_time_ms = None  # plot_unit falls back to linspace(-1, 1)

        plot_unit(
            session_id=session_id,
            unit_idx=unit_idx,
            neuron_id=neuron_id,
            cluster_id=cluster_id,
            spike_times_sec=st,
            amplitudes=amps,
            trial_events=trial_events,
            fr_similarity=fr_sim,
            isi_ratio=isi_r,
            nc_pass=nc_pass,
            nc_value=nc_val,
            acg_bins=acg_bins,
            acg_counts=acg_counts,
            conf_matrix=conf_mat,
            rp_range_ms=rp_ax,
            cont_range=cont_ax,
            wf_array=wf_array,
            channels=channels,
            chan_pos=d['chan_pos'],
            chan_map=d['chan_map'],
            wf_time_ms=wf_time_ms,
            before=before,
            after=after,
            n_units_total=len(cluster_ids),
            output_dir=str(output_dir) if output_dir is not None else None,
        )

        stable = (not np.isnan(fr_sim)) and fr_sim > 0
        isi_ok = (not np.isnan(isi_r)) and isi_r < 0.02
        summary_rows.append({
            'neuron_id':         neuron_id,
            'cluster_id':        cluster_id,
            'n_spikes':          len(st),
            'mean_fr_hz':        round(len(st) / (st.max() - st.min()), 2),
            'fr_similarity':     round(fr_sim, 4) if not np.isnan(fr_sim) else np.nan,
            'stability_pass':    stable,
            'isi_violation_pct': round(isi_r * 100, 3) if not np.isnan(isi_r) else np.nan,
            'isi_pass':          isi_ok,
            'noise_cutoff_pct':  round(nc_val, 2) if not np.isnan(nc_val) else np.nan,
            'nc_pass':           nc_pass,
            'overall_pass':      stable and isi_ok and nc_pass,
        })
        print(f"done  [overall: {fmt_pass(stable and isi_ok and nc_pass)}]")

    summary_df = pd.DataFrame(summary_rows)

    if output_dir is not None:
        csv_path = Path(output_dir) / 'qc_summary.csv'
        summary_df.to_csv(csv_path, index=False)
        print(f"\nDone. {len(summary_rows)} units processed.")
        print(f"Summary CSV : {csv_path}")
        print(f"Figures     : {output_dir}")
    return summary_df


In [ ]:
session_id = "241223_GP_TZ"
# session_id = '210126_GP_JP'
run_qc(session_id=session_id, output_dir=None, before=2.0, after=2.0)